<center>
    <a href="https://www.aus.edu/"><img src="https://i.imgur.com/pdZvnSD.png" width=200> </a>    
</center>
<h1 align=center><font size = 5>CMP 49412 - Non-Personalized Recommendations - Association Rule Mining</font>
<h1 align=center><font size = 5><b>Name:</b> Mohamed Alawadhi</font>
<h1 align=center><font size = 5><b>ID:</b> b00094286</font>

In [238]:
# Importing the necessary libraries for this assignment
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [239]:
# Read the csv file
data_df = pd.read_csv('user_ratings.csv')

In [240]:
# Print out the first 5 entires of the dataset
data_df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,5,1,4.0,847434962,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,7,1,4.5,1106635946,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,15,1,2.5,1510577970,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,17,1,4.5,1305696483,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy


In [241]:
# Printing out the shape and number of unique movies in the dataset
print(f'The shape of the dataset: {data_df.shape} \n')
print(f'There are a total of {data_df['title'].nunique()} unique movies in our dataset.')

The shape of the dataset: (100836, 6) 

There are a total of 9719 unique movies in our dataset.


**Question 1**
---

$$
\text{Bayesian Scoring} = \frac{C \times m + \sum r_i}{C + n}
$$

where:
- $C$ = Mean number of ratings across all movies (confidence factor)
- $m$ = Global average rating across all movies
- $\sum r_i$ = Sum of all ratings for the movie
- $n$ = Number of ratings for the movie

In [242]:
# Groups the dataset by the movies title in order to calculate both the 'avg_rating' (gets the mean for each movie) and num_ratings (total number of rating for each movie)
movies_avg_ratings = data_df.groupby(['title']).agg(
    avg_rating=('rating', 'mean'),
    num_ratings=('rating', 'count')
).reset_index()

In [243]:
# Calculates the global average and confidence factor
global_avg = data_df['rating'].mean()
C = data_df['title'].value_counts().median()

In [244]:
print(f'Global Average Rating: {global_avg}')
print(f'Confidence: {C}')

Global Average Rating: 3.501556983616962
Confidence: 3.0


In [245]:
movies_avg_ratings.head()

,title,avg_rating,num_ratings
0,'71 (2014),4.0,1
1,'Hellboy': The Seeds of Creation (2004),4.0,1
2,'Round Midnight (1986),3.5,2
3,'Salem's Lot (2004),5.0,1
4,'Til There Was You (1997),4.0,2


In [246]:
# Bayesian Score formula and calculation
movies_avg_ratings['bayesian_score'] = (
    (C * global_avg + movies_avg_ratings['num_ratings'] * movies_avg_ratings['avg_rating']) /
    (C + movies_avg_ratings['num_ratings'])
)

In [247]:
movies_avg_ratings.head()

,title,avg_rating,num_ratings,bayesian_score
0,'71 (2014),4.0,1,3.626168
1,'Hellboy': The Seeds of Creation (2004),4.0,1,3.626168
2,'Round Midnight (1986),3.5,2,3.500934
3,'Salem's Lot (2004),5.0,1,3.876168
4,'Til There Was You (1997),4.0,2,3.700934


In [248]:
# Sorting based on bayesian scores in a descending order and adding the rank of each movie based on the score 
movies_avg_ratings_ranked = movies_avg_ratings.sort_values(by=['bayesian_score'], ascending=False)
movies_avg_ratings_ranked['rank'] = range(1, len(movies_avg_ratings_ranked) + 1)

In [249]:
movies_avg_ratings_ranked.head()

,title,avg_rating,num_ratings,bayesian_score,rank
7593,"Shawshank Redemption, The (1994)",4.429022,317,4.420327,1
8710,"Three Billboards Outside Ebbing, Missouri (2017)",4.750000,8,4.409516,2
8917,"Trial, The (Procès, Le) (1962)",4.900000,5,4.375584,3
7486,Secrets & Lies (1996),4.590909,11,4.357476,4
8110,"Streetcar Named Desire, A (1951)",4.475000,20,4.348029,5


In [250]:
# Displaying the top 5 movies for Bayesian Scoring
top_5_movies = movies_avg_ratings_ranked.head()
print("The top 5 movies found using Bayesian Scoring:-")
display(top_5_movies[['title', 'bayesian_score', 'rank']])

The top 5 movies found using Bayesian Scoring:-


,title,bayesian_score,rank
7593,"Shawshank Redemption, The (1994)",4.420327,1
8710,"Three Billboards Outside Ebbing, Missouri (2017)",4.409516,2
8917,"Trial, The (Procès, Le) (1962)",4.375584,3
7486,Secrets & Lies (1996),4.357476,4
8110,"Streetcar Named Desire, A (1951)",4.348029,5


**Question 2**
---

$$
\text{Wilson Scoring} = \frac{p + \frac{z^2}{2\times n} - z \sqrt{\frac{p(1-p) + \frac{z^2}{4\times n}}{n}}}{1 + \frac{z^2}{n}}
$$

where:
- $p$ = Thumbs Up Ratio
- $n$ = Total Votes
- $z$ = Confidence Level

In [251]:
# Reading the csv file again in a new variable and converting 'rating' into 1 (thumbs up) and 0 (thumbs down) 
data_df_wilson = pd.read_csv('user_ratings.csv')
data_df_wilson['rating'] = data_df_wilson['rating'].apply(lambda x: 1 if x >= 4.0 else 0)

In [252]:
data_df_wilson.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,1,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,5,1,1,847434962,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,7,1,1,1106635946,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,15,1,0,1510577970,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,17,1,1,1305696483,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy


In [253]:
movies_votes = data_df_wilson.groupby(['movieId', 'title']).agg(
    thumbs_up=('rating', 'sum'),
    total_votes=('rating', 'count')
).reset_index()

movies_votes['thumbs_down'] = movies_votes['total_votes'] - movies_votes['thumbs_up']
movies_votes = movies_votes.iloc[:, [0, 1, 2, 4, 3]]    # Swaps positions between 'total_votes' and 'thumbs_down'

In [254]:
movies_votes.head()

,movieId,title,thumbs_up,thumbs_down,total_votes
0,1,Toy Story (1995),147,68,215
1,2,Jumanji (1995),50,60,110
2,3,Grumpier Old Men (1995),18,34,52
3,4,Waiting to Exhale (1995),0,7,7
4,5,Father of the Bride Part II (1995),12,37,49


In [255]:
z = 1.96 # Most commonly used (95% confidence)
p = movies_votes['thumbs_up'] / movies_votes['total_votes']
n = movies_votes['total_votes']

In [256]:
movies_votes['p_ratio'] = p
movies_votes.head()

,movieId,title,thumbs_up,thumbs_down,total_votes,p_ratio
0,1,Toy Story (1995),147,68,215,0.683721
1,2,Jumanji (1995),50,60,110,0.454545
2,3,Grumpier Old Men (1995),18,34,52,0.346154
3,4,Waiting to Exhale (1995),0,7,7,0.000000
4,5,Father of the Bride Part II (1995),12,37,49,0.244898


In [257]:
# Wilson Score formula and calculation
movies_votes['wilson_score'] = ((p + (z**2 / (2*n)) 
                                - z * np.sqrt((p*(1-p) + z**2 / (4*n)) / n)) /
                                (1 + (z**2 / n))
                                )

In [258]:
movies_votes.head()

,movieId,title,thumbs_up,thumbs_down,total_votes,p_ratio,wilson_score
0,1,Toy Story (1995),147,68,215,0.683721,0.618799
1,2,Jumanji (1995),50,60,110,0.454545,0.364598
2,3,Grumpier Old Men (1995),18,34,52,0.346154,0.231508
3,4,Waiting to Exhale (1995),0,7,7,0.000000,0.000000
4,5,Father of the Bride Part II (1995),12,37,49,0.244898,0.146022


In [259]:
movies_ranked = movies_votes.sort_values(by=['wilson_score'], ascending=False)
movies_ranked['rank'] = range(1, len(movies_votes) + 1)

In [260]:
# Displaying the top 5 movies based on the Wilson Scores of each movie
print("The top 5 movies found using Wilson Scoring:-")
display(movies_ranked.head())

The top 5 movies found using Wilson Scoring:-


,movieId,title,thumbs_up,thumbs_down,total_votes,p_ratio,wilson_score,rank
277,318,"Shawshank Redemption, The (1994)",274,43,317,0.864353,0.822270,1
2224,2959,Fight Club (1999),179,39,218,0.821101,0.764799,2
711,930,Notorious (1946),19,1,20,0.950000,0.763864,3
659,858,"Godfather, The (1972)",158,34,192,0.822917,0.762743,4
6298,48516,"Departed, The (2006)",90,17,107,0.841121,0.760223,5


**Question 3**
---

In [261]:
data_df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,5,1,4.0,847434962,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,7,1,4.5,1106635946,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,15,1,2.5,1510577970,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,17,1,4.5,1305696483,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy


In [262]:
# Adding 'watched' to the dataset if rating >= 4
data_df['watched'] = data_df['rating'].apply(lambda x: 1 if x >= 4.0 else 0)

In [263]:
data_df.head()

,userId,movieId,rating,timestamp,title,genres,watched
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1
1,5,1,4.0,847434962,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1
2,7,1,4.5,1106635946,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1
3,15,1,2.5,1510577970,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,0
4,17,1,4.5,1305696483,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1


In [264]:
# Filters the dataset with 'watched' being true
watched_df = data_df[data_df['watched'] == 1]

In [265]:
print(f'The shape of the watched dataset: {watched_df.shape} \n')
print(f'There are a total of {watched_df['title'].nunique()} unique movies in the watched dataset.')

The shape of the watched dataset: (48580, 7) 

There are a total of 6297 unique movies in the watched dataset.


In [266]:
# Creates a table of movie titles and if a user watched those movies or not 
transactions = watched_df.pivot_table(
    index='userId', columns='title', values='watched', fill_value=0
)

In [267]:
transactions.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Salem's Lot (2004),'Til There Was You (1997),"'burbs, The (1989)",(500) Days of Summer (2009),*batteries not included (1987),...And Justice for All (1979),00 Schneider - Jagd auf Nihil Baxter (1994),1-900 (06) (1994),...,Zombieland (2009),Zookeeper (2011),Zoolander (2001),Zootopia (2016),Zulu (1964),[REC] (2007),[REC]² (2009),eXistenZ (1999),xXx (2002),¡Three Amigos! (1986)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [268]:
frequent_itemsets = apriori(transactions, min_support=0.2, use_colnames=True)

c:\Users\Oroch\anaconda3\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [269]:
frequent_itemsets.head()

,support,itemsets
0,0.249589,(American Beauty (1999))
1,0.208539,(Apollo 13 (1995))
2,0.272578,(Braveheart (1995))
3,0.224959,(Fargo (1996))
4,0.293924,(Fight Club (1999))


In [270]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=2.0)

In [271]:
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Star Wars: Episode IV - A New Hope (1977)),(Star Wars: Episode V - The Empire Strikes Bac...,0.330049,0.275862,0.236453,0.716418,2.597015,1.0,0.145405,2.553539,0.917892,0.64,0.608387,0.786780
1,(Star Wars: Episode V - The Empire Strikes Bac...,(Star Wars: Episode IV - A New Hope (1977)),0.275862,0.330049,0.236453,0.857143,2.597015,1.0,0.145405,4.689655,0.849206,0.64,0.786765,0.786780
2,(Star Wars: Episode VI - Return of the Jedi (1...,(Star Wars: Episode IV - A New Hope (1977)),0.246305,0.330049,0.206897,0.840000,2.545075,1.0,0.125604,4.187192,0.805478,0.56,0.761176,0.733433
3,(Star Wars: Episode IV - A New Hope (1977)),(Star Wars: Episode VI - Return of the Jedi (1...,0.330049,0.246305,0.206897,0.626866,2.545075,1.0,0.125604,2.019901,0.906162,0.56,0.504926,0.733433


In [272]:
top_rules = rules.sort_values(by="confidence", ascending=False).head(3)

print("Top 3 Association Rules:")
for _, row in top_rules.iterrows():
    print(f"Users who watched {list(row['antecedents'])} also watched {list(row['consequents'])} \n"
          f"-support={row['support']:.2f}, confidence={row['confidence']:.2f}, lift={row['lift']:.2f}-\n")

Top 3 Association Rules:
Users who watched ['Star Wars: Episode V - The Empire Strikes Back (1980)'] also watched ['Star Wars: Episode IV - A New Hope (1977)'] 
-support=0.24, confidence=0.86, lift=2.60-

Users who watched ['Star Wars: Episode VI - Return of the Jedi (1983)'] also watched ['Star Wars: Episode IV - A New Hope (1977)'] 
-support=0.21, confidence=0.84, lift=2.55-

Users who watched ['Star Wars: Episode IV - A New Hope (1977)'] also watched ['Star Wars: Episode V - The Empire Strikes Back (1980)'] 
-support=0.24, confidence=0.72, lift=2.60-

